In [92]:
from datetime import datetime
import GRANTA_MIScriptingToolkit as gdl
from GRANTA_MIScriptingToolkit import granta as mpy
import pathlib 
import numpy as np

In [93]:
usernameVal = 'STK_user'
passwordVal = 'P455w07d!'

session = mpy.connect('https://nanomecommons.grantami.com/mi_servicelayer', user_name=usernameVal, password=passwordVal)

mi = mpy.connect('https://nanomecommons.grantami.com/mi_servicelayer', user_name=usernameVal, password=passwordVal)
dbKey = "MI_Blank_v2021"
db = session.get_db(db_key=dbKey)
db = mi.get_db(db_key=dbKey)
test_table = db.get_table("Indentation test data")
folderName ='Python Demo'

In [94]:
# Import the data from the Excel file
file_path = '3D_PrA-_IndProf-NanoBlitz-VR1.xlsx'
sheet_name = 'Test 1'
data = pd.read_excel(file_path, sheet_name=sheet_name, header=0, skiprows=[1])
print(data.columns)

Index(['Markers', 'Index', 'X Position', 'Y Position', 'MODULUS', 'HARDNESS'], dtype='object')


In [95]:
now = datetime.now().strftime("%c")
recordName = 'BASF_VR1: {}'.format(now)
rec = test_table.create_record(recordName)

In [96]:
attribName = "Mechanical properties matrix"
hist = rec.attributes[attribName]
hist
hist.show()


          X position           |           Y position           |           Z position           |     Young's modulus value      |         Hardness value         |             Creep              |      Linking Value (None)     
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------



In [97]:
# Prefill the 'Z Position' column with zeros if it is empty
if 'Z Position' not in data.columns or data['Z Position'].isnull().all():
    data['Z Position'] = 0

# Prefill the 'Z Position' column with zeros if it is empty
if 'Creep' not in data.columns or data['Creep'].isnull().all():
    data['Creep'] = 0

# Create a new DataFrame with the specified column names
new_data = pd.DataFrame({
    
    'X position': data['X Position'],
    'Y position': data['Y Position'],
    'Z position': data['Z Position'],
    'Young\'s modulus value': data['MODULUS'],
    'Hardness value': data['HARDNESS'],
    'Creep': data['Creep'] if 'Creep' in data.columns else None
})

# Save the new DataFrame to a CSV file
new_data.to_csv('example.csv', index=False)

print("Data has been successfully saved to example.csv")
print(new_data.head())

Data has been successfully saved to example.csv
   X position  Y position  Z position  Young's modulus value  Hardness value  \
0         0.0   10.763250           0               1.645402        0.079737   
1         0.0    9.572625           0               1.797390        0.088518   
2         0.0    8.382000           0               1.861923        0.092811   
3         0.0    7.191375           0               1.856298        0.098128   
4         0.0    5.953125           0               1.758753        0.088199   

   Creep  
0      0  
1      0  
2      0  
3      0  
4      0  


In [98]:
print("Columns in the CSV file:", data.columns)

Columns in the CSV file: Index(['Markers', 'Index', 'X Position', 'Y Position', 'MODULUS', 'HARDNESS',
       'Z Position', 'Creep'],
      dtype='object')


In [99]:
for index, row in data.iterrows():
    hist.add_row()
    hist['X position', index] = new_data['X position'][index]
    hist['Y position', index] = new_data['Y position'][index] 
    hist['Z position', index] = new_data['Z position'][index]
    hist['Young\'s modulus value', index] = new_data['Young\'s modulus value'][index]
    hist['Hardness value', index] = new_data['Hardness value'][index]
    hist['Creep', index] = new_data['Creep'][index]
    
hist.value[:][0]

[0.0,
 10.7632499999968,
 0,
 1.64540152497631,
 0.0797367357526049,
 0,
 '<linking value not chosen>']

In [100]:
hist.show()

          X position           |           Y position           |           Z position           |     Young's modulus value      |         Hardness value         |             Creep              |      Linking Value (None)     
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
             0.0               |        10.7632499999968        |               0                |        1.64540152497631        |       0.0797367357526049       |               0                |  '<linking value not chosen>' 
             0.0               |        9.57262499999556        |               0                |        1.79739046074821        |       0.0885175905320609       |               0                |  '<linking value not chosen>' 
             0.0               |        8.38199999999428        |               0   

In [101]:
rec.set_attributes([hist])
rec = mi.update([rec])[0]